In [1]:
import MySQLdb

In [2]:
class Word:
  def __init__(self, eng, kor, lev="1"):
    self.eng = eng
    self.kor = kor
    self.lev = lev
  
  def __repr__(self):
    return f"Word(eng='{self.eng}', kor='{self.kor}', lev='{self.lev}')"

  @property
  def eng(self):
    return self.__eng
  
  @eng.setter
  def eng(self, eng):
    if not eng:
      raise ValueError("영어 단어는 비워둘 수 없습니다")
    self.__eng = eng
  
  @property
  def kor(self):
    return self.__kor
  
  @kor.setter
  def kor(self, kor):
    if not kor:
      raise ValueError("뜻을 비워둘 수 없습니다")
    self.__kor = kor
  
  @property
  def lev(self):
    return self.__lev
  
  @lev.setter
  def lev(self, lev):
    lev = int(lev)
    if lev < 1:
      raise ValueError("레벨은 1 이상이여야 합니다.")
    self.__lev = lev
  

In [16]:
class WordsDAO:
  def __init__(self):
    self.db = None
  
  def connect(self):
    self.db = MySQLdb.connect(host='localhost', user='root', password='1234', db='ai', charset='utf8')

  def disconnect(self):
    if self.db:
      self.db.close()
  
  def insert(self, word: Word) -> None:
    self.connect()
    cur = self.db.cursor()
    sql = "insert into voca (eng, kor, lev) values (%s, %s ,%s)"
    data = (word.eng, word.kor, word.lev)
    cur.execute(sql, data)
    self.db.commit()
    cur.close()
    self.disconnect()
  
  def search(self, eng):
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)
        sql = "select eng, kor, lev from voca where eng like concat('%%', %s, '%%')"
        cur.execute(sql, (eng,))
        rows = cur.fetchall()
        cur.close()
        self.disconnect()
        return rows

  def select_all(self) -> dict:
    self.connect()
    cur = self.db.cursor(MySQLdb.cursors.DictCursor)
    sql = "select eng, kor, lev from voca order by eng asc"
    cur.execute(sql)
    rows = cur.fetchall()
    cur.close()
    self.disconnect()
    return rows

  def update(self, word):
        self.connect()
        cur = self.db.cursor()
        data = (word.kor, word.lev, word.eng)
        print(data)
        sql = "update voca set kor=%s, lev=%s where eng = %s"
        result = cur.execute(sql, data)
        self.db.commit()
        cur.close()
        self.disconnect()
        return result
  def delete(self, eng):
        self.connect()
        cur = self.db.cursor()
        sql = "delete from voca where eng = %s"
        result = cur.execute(sql, (eng,))
        self.db.commit()
        cur.close()
        self.disconnect()
        return result


In [4]:
dao = WordsDAO()
dao.connect()
dao.disconnect()

In [5]:
class WordsService:
  def __init__(self):
    self.dao = WordsDAO()
  
  def insert_word(self):
    eng = input("단어를 입력하세요: ")
    kor = input("뜻을 입력하세요: ")
    lev = input("레벨을 입력하세요: ")

    word = Word(eng, kor, lev)
    self.dao.insert(word)
    print("단어가 등록되었습니다")
    
  def print_all(self):
    words = self.dao.select_all()
    if not words:
      print("등록된 단어가 없습니다")
      return
    
    for word in words:
      print(f"{word['eng']}: {word['kor']}   | lev: {word['lev']}")
  
  def search_words(self):
    eng = input("검색할 단어를 입력하세요: ")
    words = self.dao.search(eng)
    if not words:
        print("찾는 단어가 없습니다")
        return
    for word in words:
        print(f"{word['eng']} 뜻:{word['kor']}, 레벨:{word['lev']}")
  
  def edit_word(self):
    eng = input("수정할 단어를 입력하세요: ")
    word = self.dao.search(eng)
    print(word)
    if not word:
      print("수정할 단어가 없습니다")
      return
    kor = input("새로운 뜻을 입력하세요: ")
    lev = input("새로운 레벨을 입력하세요: ")
    word = Word(eng, kor, lev)
    result = self.dao.update(word)
    if result > 0:
      print('수정되었습니다')
    else:
      print('수정에 실패했습니다')
      
  def delete_word(self):
        eng = input("삭제할 단어를 입력하세요: ")
        word = self.dao.search(eng)
        if not word:
            print("삭제할 단어가 없습니다")
            return
        result = self.dao.delete(eng)
        if result > 0:
            print("삭제되었습니다")
        else:
            print("삭제에 실패했습니다")

In [6]:
class Menu:
  def __init__(self):
    self.service = WordsService()

  def run(self):
    while True:
      try:
        print()
        print("===== 단어장 프로그램 =====")
        print("1. 단어 등록하기")
        print("2. 단어 출력하기")
        print("3. 단어 검색하기")
        print("4. 단어 수정하기")
        print("5. 단어 삭제하기")
        print("6. 종료하기")

        try:
          menu = int(input("메뉴를 선택하세요 (숫자로)"))
        except Exception as e:
          print("숫자를 입력해주세요")
          continue

        if menu == 1:
          print("등록합니다")
          self.service.insert_word()
          
        elif menu == 2:
          print("출력합니다")
          self.service.print_all()
          
        elif menu == 3:
          print("검색합니다")
          self.service.search_words()

        elif menu == 4:
          print("수정합니다")
          self.service.edit_word()

        elif menu == 5:
          print("삭제합니다")
          self.service.delete_word()
          
        elif menu == 6:
          print("프로그램을 종료합니다")
          break
        else:
          print("메뉴는 1부터 6만 존재합니다")
      except Exception as e:
        print("오류:",e)

In [17]:
menu = Menu()
menu.run()


===== 단어장 프로그램 =====
1. 단어 등록하기
2. 단어 출력하기
3. 단어 검색하기
4. 단어 수정하기
5. 단어 삭제하기
6. 종료하기
수정합니다
({'eng': 'mongo', 'kor': '몽고', 'lev': 1},)
('뭉치', 12, 'mongo')
수정되었습니다

===== 단어장 프로그램 =====
1. 단어 등록하기
2. 단어 출력하기
3. 단어 검색하기
4. 단어 수정하기
5. 단어 삭제하기
6. 종료하기
출력합니다
apple: 사과   | lev: 1
banana: 바나나   | lev: 1
kiwi: 키위   | lev: 1
mongo: 뭉치   | lev: 12

===== 단어장 프로그램 =====
1. 단어 등록하기
2. 단어 출력하기
3. 단어 검색하기
4. 단어 수정하기
5. 단어 삭제하기
6. 종료하기
숫자를 입력해주세요

===== 단어장 프로그램 =====
1. 단어 등록하기
2. 단어 출력하기
3. 단어 검색하기
4. 단어 수정하기
5. 단어 삭제하기
6. 종료하기
숫자를 입력해주세요

===== 단어장 프로그램 =====
1. 단어 등록하기
2. 단어 출력하기
3. 단어 검색하기
4. 단어 수정하기
5. 단어 삭제하기
6. 종료하기
숫자를 입력해주세요

===== 단어장 프로그램 =====
1. 단어 등록하기
2. 단어 출력하기
3. 단어 검색하기
4. 단어 수정하기
5. 단어 삭제하기
6. 종료하기
프로그램을 종료합니다
